# 04 連續分布（講義 05、HW05）

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats

import statkit as sk
sk.setup()  # 字體大小 16、中文字型

## 自訂 pdf：用 sympy 求 k、機率、E[X]、Var(X)、F(x)

講義例：$f(x) = kx^2$，$x \in [0, 1]$。答案：$k = 3$、$\Pr(X \ge \tfrac12) = \tfrac78$、$E[X] = \tfrac34$、$\operatorname{Var}(X) = \tfrac{3}{80}$

In [ ]:
x, k, t = sp.symbols('x k t', positive=True)
f = k * x ** 2
k_val = sp.solve(sp.Eq(sp.integrate(f, (x, 0, 1)), 1), k)[0]
f = f.subs(k, k_val)
prob = sp.integrate(f, (x, sp.Rational(1, 2), 1))
EX = sp.integrate(x * f, (x, 0, 1))
VarX = sp.integrate((x - EX) ** 2 * f, (x, 0, 1))
F = sp.integrate(f.subs(x, t), (t, 0, x))
print('k =', k_val, ' Pr(X >= 1/2) =', prob, ' E[X] =', EX, ' Var(X) =', VarX)
print('F(x) =', F)

分段 pdf（HW05）：分段積分再相加

In [ ]:
x, k = sp.symbols('x k', positive=True)
total = sp.integrate(k * x ** 4, (x, 0, 1)) + sp.integrate(k * (4 - x) ** 12, (x, 1, 4))
k_val = sp.solve(sp.Eq(total, 1), k)[0]
EX = sp.integrate(x * k_val * x ** 4, (x, 0, 1)) + sp.integrate(x * k_val * (4 - x) ** 12, (x, 1, 4))
print('k =', sp.simplify(k_val), float(k_val), ' E[X] =', float(EX))

數值積分（不需要精確解時）：

In [ ]:
from scipy import integrate
value, err = integrate.quad(lambda x: 3 * x ** 2, 0.5, 1)
print(value)

## 均勻分布 Uni(a, b)

$f(x) = \frac{1}{b-a}$，$F(x) = \frac{x-a}{b-a}$，$E[X] = \frac{a+b}{2}$，$\operatorname{Var}(X) = \frac{(b-a)^2}{12}$。scipy：`loc=a, scale=b-a`

In [ ]:
a, b = 4.7, 5.3    # 講義：一箱橘子的重量
print('E[X] =', stats.uniform.mean(loc=a, scale=b - a), ' Var(X) =', round(stats.uniform.var(loc=a, scale=b - a), 4))
print('Pr(X >= 15) =', round(1 - stats.uniform.cdf(x=15, loc=10, scale=6), 4))
print('x for Pr(X >= x) = 0.01:', stats.uniform.ppf(q=1 - 0.01, loc=10, scale=6))
sk.Uniform(10, 16).P('X > 15')

## 指數分布 Exp(λ)

$f(x) = \lambda e^{-\lambda x}$，$F(x) = 1 - e^{-\lambda x}$，$E[X] = 1/\lambda$，$\operatorname{Var}(X) = 1/\lambda^2$。

⚠ **scipy 用 `scale = 1/λ`**（λ 是發生率）。卜瓦松：每單位時間幾次；指數：兩次之間隔多久。

In [ ]:
lam = 6.6    # 每小時 6.6 次
print('Pr(X > 25 min) =', round(1 - stats.expon.cdf(x=25 / 60, scale=1 / lam), 4))
print('Pr(X <= 5 min) =', round(stats.expon.cdf(x=5 / 60, scale=1 / lam), 4))
sk.Expon(lam).P('X > 25/60')
sk.plot_dist(sk.Expon(lam), 'X > 25/60', xlabel='hours')
plt.show()

## 卜瓦松與指數：次數與等待時間

- 每單位時間平均 λ 次 → 次數 $\sim \text{Poi}(\lambda)$，兩次之間的間隔 $\sim \text{Exp}(\lambda)$，平均間隔 $\frac{1}{\lambda}$
- 第 k 位到第 m 位之間有 $m - k$ 段間隔 → 期望時間 $\frac{m - k}{\lambda}$
- 時間單位要一致：λ 是「每小時」，算 10 分鐘就寫 `10 / 60`

In [ ]:
lam = 8      # 每小時平均 8 位
print('一小時內至少 5 位：', round(1 - stats.poisson.cdf(k=4, mu=lam), 4))
print('一小時內剛好 8 位：', round(stats.poisson.pmf(k=8, mu=lam), 4))
print('第 3 位到第 7 位之間的期望時間：', (7 - 3) / lam, '小時 =', (7 - 3) / lam * 60, '分鐘')
print('下一位在 10 分鐘內來的機率：', round(stats.expon.cdf(x=10 / 60, scale=1 / lam), 4))

## 無記憶性、計程車或公車（講義 05）

$\Pr(X > s + v \mid X > s) = \Pr(X > v)$。公車每小時 5 班 → 每分鐘 λ = 1/12：

In [ ]:
p = 1 - stats.expon.cdf(x=10, scale=12)      # Pr(X > 10 分鐘)
print('p =', round(p, 4), '→ 期望成本', round(20 * (1 - p) + 320 * p, 2), 'vs 計程車 200')
for s in [1, 2, 3, 4]:
    q = (1 - stats.expon.cdf(x=10, scale=12)) / (1 - stats.expon.cdf(x=s, scale=12))
    print(f'已等 {s} 分鐘：Pr(T > {10 - s}) = {q:.4f}', '→ 改搭計程車' if q >= 0.6 else '')

## 報童模型（Newsvendor）

$c_o = c + d - v$（多訂）、$c_u = r - c + s$（少訂），最適訂購量 $F(q^*) = \frac{c_u}{c_o + c_u}$。
講義例：成本 5、售價 15、需求 Uni(20, 50) → $q^* = 40$。

In [ ]:
co, cu = 5, 15 - 5
q_star = stats.uniform.ppf(q=cu / (co + cu), loc=20, scale=30)
print('q* =', q_star)
sk.newsvendor(sk.Uniform(20, 50), co=4, cu=11)     # 成本降到 4 → 42
sk.newsvendor(sk.Normal(100, 15), co=2, cu=8)       # 常態需求也可以

## PDF / CDF 圖（課程寫法）

In [ ]:
xs = np.linspace(0, 1, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(xs, stats.expon.pdf(xs, scale=1 / 6.6), color='black')
axes[0].set_title('pdf of Exp(6.6)')
axes[1].plot(xs, stats.expon.cdf(xs, scale=1 / 6.6), color='black')
axes[1].set_title('cdf of Exp(6.6)')
plt.show()